<a href="https://colab.research.google.com/github/yahavan/me526_assignments/blob/main/data_analysis_assignment_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Foundations of Statistical Inference, PCA & Factor Analysis — Worked Solutions

This notebook answers every part of the assignment: theoretical derivations (rendered in LaTeX),
from-scratch numerical verification, and the interactive Plotly diagnostic dashboards.

**Run order:** top to bottom. The setup cell below installs the only non-default Colab dependency.

> Sections
> 1. Statistical Inference & Hypothesis Testing (MLE bias, Type I/II, Slutsky, MANOVA / Wilks' Λ)
> 2. Geometric Subspace Optimization via PCA ($T^2$ & $Q$)
> 3. Latent Subspace Decomposition via Factor Analysis (Thomson scores, Varimax, dashboard)
> 4. Subspace Diagnostics & Feature De-correlation (dual dashboards + analysis)

In [ ]:
# --- Environment setup (Colab) ---
# statsmodels, scipy, numpy, pandas, scikit-learn, plotly ship with Colab.
# This is a no-op safety install in case a clean kernel is used.
try:
    import statsmodels, plotly, sklearn  # noqa
except Exception:
    !pip -q install statsmodels plotly scikit-learn

import numpy as np
import pandas as pd
import scipy.stats as stats
np.set_printoptions(precision=4, suppress=True)
print("Setup ready.")

# Section 1 — Foundations of Statistical Inference & Hypothesis Testing

## Part A — Theoretical Fundamentals

### A.1  Bias of the MLE covariance and Bessel's correction

**Claim.** For i.i.d. $\mathbf X_1,\dots,\mathbf X_n\in\mathbb R^m$ with mean $\boldsymbol\mu$ and covariance $\boldsymbol\Sigma$, the raw MLE
$$\widehat{\boldsymbol\Sigma}_{\text{MLE}}=\frac1n\sum_{i=1}^n(\mathbf X_i-\widehat{\boldsymbol\mu}_n)(\mathbf X_i-\widehat{\boldsymbol\mu}_n)^{\!\top},\qquad \widehat{\boldsymbol\mu}_n=\frac1n\sum_{i=1}^n\mathbf X_i$$
satisfies $\;\mathbb E[\widehat{\boldsymbol\Sigma}_{\text{MLE}}]=\dfrac{n-1}{n}\boldsymbol\Sigma.$

**Proof.** Center the data with $\mathbf Y_i=\mathbf X_i-\boldsymbol\mu$, so $\mathbb E[\mathbf Y_i]=\mathbf 0$, $\mathbb E[\mathbf Y_i\mathbf Y_i^{\!\top}]=\boldsymbol\Sigma$, and write $\bar{\mathbf Y}=\widehat{\boldsymbol\mu}_n-\boldsymbol\mu=\tfrac1n\sum_i\mathbf Y_i$. Because covariance is translation-invariant,
$$\sum_{i=1}^n(\mathbf X_i-\widehat{\boldsymbol\mu}_n)(\mathbf X_i-\widehat{\boldsymbol\mu}_n)^{\!\top}=\sum_{i=1}^n(\mathbf Y_i-\bar{\mathbf Y})(\mathbf Y_i-\bar{\mathbf Y})^{\!\top}.$$
Expanding and using $\sum_i\mathbf Y_i=n\bar{\mathbf Y}$ gives the standard identity
$$\sum_{i=1}^n(\mathbf Y_i-\bar{\mathbf Y})(\mathbf Y_i-\bar{\mathbf Y})^{\!\top}=\sum_{i=1}^n\mathbf Y_i\mathbf Y_i^{\!\top}-n\,\bar{\mathbf Y}\bar{\mathbf Y}^{\!\top}.$$
Take expectations term by term:
$$\mathbb E\!\Big[\sum_i\mathbf Y_i\mathbf Y_i^{\!\top}\Big]=n\boldsymbol\Sigma,\qquad
\mathbb E[\bar{\mathbf Y}\bar{\mathbf Y}^{\!\top}]=\operatorname{Var}(\bar{\mathbf Y})=\frac1{n^2}\sum_i\operatorname{Var}(\mathbf Y_i)=\frac1n\boldsymbol\Sigma,$$
the last step using i.i.d. independence so cross-terms vanish. Hence
$$\mathbb E\!\Big[\sum_i(\mathbf Y_i-\bar{\mathbf Y})(\mathbf Y_i-\bar{\mathbf Y})^{\!\top}\Big]=n\boldsymbol\Sigma-n\cdot\tfrac1n\boldsymbol\Sigma=(n-1)\boldsymbol\Sigma,$$
$$\boxed{\;\mathbb E[\widehat{\boldsymbol\Sigma}_{\text{MLE}}]=\frac1n(n-1)\boldsymbol\Sigma=\frac{n-1}{n}\boldsymbol\Sigma.\;}$$
The estimator systematically *under-estimates* $\boldsymbol\Sigma$ by the factor $\tfrac{n-1}{n}<1$; one degree of freedom is consumed estimating $\widehat{\boldsymbol\mu}_n$ from the same sample.

**Bessel's correction → unbiased $\mathbf S$.** Rescale by $\tfrac{n}{\,n-1\,}$:
$$\mathbf S=\frac1{n-1}\sum_{i=1}^n(\mathbf X_i-\widehat{\boldsymbol\mu}_n)(\mathbf X_i-\widehat{\boldsymbol\mu}_n)^{\!\top}=\frac{n}{n-1}\,\widehat{\boldsymbol\Sigma}_{\text{MLE}}
\;\Longrightarrow\;
\mathbb E[\mathbf S]=\frac{n}{n-1}\cdot\frac{n-1}{n}\boldsymbol\Sigma=\boldsymbol\Sigma.$$
**Uniqueness.** Any estimator of the form $c\sum_i(\mathbf X_i-\widehat{\boldsymbol\mu}_n)(\cdots)^{\!\top}$ has expectation $c(n-1)\boldsymbol\Sigma$; unbiasedness forces $c(n-1)=1$, i.e. $c=\tfrac1{n-1}$ uniquely. Thus $\mathbf S$ is the unique scalar-rescaled unbiased estimator.

### A.2  Type I / Type II error and an ultra-conservative threshold

**Type I error $(\alpha)$ — false alarm.** Rejecting $H_0$ (declaring damage / anomaly) when the asset is in fact healthy. In SHM this halts a *healthy* plant for needless inspection. $\alpha=\Pr(\text{reject }H_0\mid H_0\text{ true})$.

**Type II error $(\beta)$ — missed detection.** Failing to reject $H_0$ (declaring "healthy") when damage is actually present. The crack propagates undetected. $\beta=\Pr(\text{fail to reject }H_0\mid H_1\text{ true})$, and **statistical power** $=1-\beta$.

**Consequence of $\alpha=10^{-4}$.** The decision rule keeps a point "healthy" while
$$D^2=(\mathbf x-\widehat{\boldsymbol\mu}_n)^{\!\top}\mathbf S^{-1}(\mathbf x-\widehat{\boldsymbol\mu}_n)\le \chi^2_{m,\,1-\alpha}.$$
Driving $\alpha$ down pushes the critical quantile $\chi^2_{m,1-\alpha}$ *up*. For a fixed true effect size and fixed $n$, the rejection region shrinks, so $\beta$ grows and **power $1-\beta$ collapses** — the diagnostic becomes nearly blind to real but subtle damage.

**Geometry of the confidence ellipsoid.** The "healthy" region is the ellipsoid $\{\mathbf x:D^2\le \chi^2_{m,1-\alpha}\}$ whose volume is
$$V=\frac{\pi^{m/2}}{\Gamma\!\big(\tfrac m2+1\big)}\,\big(\chi^2_{m,1-\alpha}\big)^{m/2}\sqrt{\det\mathbf S}.$$
Because $V\propto(\chi^2_{m,1-\alpha})^{m/2}$, shrinking $\alpha$ **inflates the ellipsoid volume** (super-linearly in $m$). The acceptance region swells and swallows genuinely anomalous snapshots — exactly the trade you make to suppress false alarms.

## Part B — Asymptotic Distributions & Slutsky's Theorem

**Slutsky's Theorem (rigorous statement).** Let $\{X_n\}$, $\{Y_n\}$ be sequences of random vectors/matrices on a common space with
$$X_n\xrightarrow{\;d\;}X,\qquad Y_n\xrightarrow{\;p\;}c\quad(c\text{ a constant}).$$
Then, for the continuous operations where the limit is well defined,
$$X_n+Y_n\xrightarrow{d}X+c,\qquad Y_nX_n\xrightarrow{d}cX,\qquad X_n/Y_n\xrightarrow{d}X/c\ \ (c\ne0).$$
Matrix form: if $A_n\xrightarrow{p}A$ (constant matrix) and $\mathbf X_n\xrightarrow{d}\mathbf X$, then $A_n\mathbf X_n\xrightarrow{d}A\mathbf X$. The key asymmetry: one sequence needs only convergence **in probability to a constant**, the other convergence **in distribution**.

**Justifying $\mathbf S$ in place of $\boldsymbol\Sigma$.** The multivariate CLT gives
$$\sqrt n(\widehat{\boldsymbol\mu}_n-\boldsymbol\mu)\xrightarrow{d}\mathscr N(\mathbf 0,\boldsymbol\Sigma).$$
The sample covariance is a consistent estimator, $\mathbf S\xrightarrow{p}\boldsymbol\Sigma$ (WLLN applied to the second moments), and since the matrix square-root/inverse are continuous on the cone of positive-definite matrices, the continuous-mapping theorem yields $\mathbf S^{-1/2}\xrightarrow{p}\boldsymbol\Sigma^{-1/2}$. Apply Slutsky with $A_n=\mathbf S^{-1/2}\xrightarrow{p}\boldsymbol\Sigma^{-1/2}$ and $\mathbf X_n=\sqrt n(\widehat{\boldsymbol\mu}_n-\boldsymbol\mu)\xrightarrow{d}\mathscr N(\mathbf 0,\boldsymbol\Sigma)$:
$$\mathbf S^{-1/2}\sqrt n(\widehat{\boldsymbol\mu}_n-\boldsymbol\mu)\xrightarrow{d}\boldsymbol\Sigma^{-1/2}\,\mathscr N(\mathbf 0,\boldsymbol\Sigma)=\mathscr N\!\big(\mathbf 0,\ \boldsymbol\Sigma^{-1/2}\boldsymbol\Sigma\,\boldsymbol\Sigma^{-1/2}\big)=\mathscr N(\mathbf 0,\mathbf I_m).$$
Reading the standardization backwards, replacing $\boldsymbol\Sigma$ by the observable $\mathbf S$ is asymptotically lossless, which operationalizes the practical sampling model
$$\boxed{\;\widehat{\boldsymbol\mu}_n\ \overset{\text{approx}}{\sim}\ \mathscr N\!\Big(\boldsymbol\mu,\ \tfrac1n\mathbf S\Big).\;}$$

## Part C — Numerical Verification (MANOVA / Wilks' $\Lambda$)

We partition the $n=5000$ snapshots into $g=5$ consecutive blocks of $n_j=1000$ and test
**Condition 1 (constant joint mean)** with Wilks' $\Lambda$ transformed through Bartlett's $\chi^2$ approximation. The function is implemented *from scratch* and then cross-checked against `statsmodels`.

In [1]:
# --- DO NOT MODIFY: Synthetic Strain Sensor Data Generator ---
np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

# Injecting a subtle, hidden structural drift in the final 1,000 snapshots
base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])
# ----------------------------------------------------------------

NameError: name 'np' is not defined

In [ ]:
def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """
    Partition into g consecutive blocks and test first-moment homogeneity via
    Wilks' Lambda + Bartlett's chi-square asymptotic transform.

        Lambda      = |W| / |B + W|
        chi2_calc   = -(n - 1 - (m+g)/2) * ln(Lambda)
        dof         = m * (g - 1)
    """
    X = df.to_numpy(dtype=float)
    n, m = X.shape
    g = g_chunks
    block = n // g

    grand_mean = X.mean(axis=0)
    W = np.zeros((m, m))   # within-chunk scatter
    B = np.zeros((m, m))   # between-chunk scatter

    for j in range(g):
        chunk = X[j * block:(j + 1) * block]
        nj = chunk.shape[0]
        cmean = chunk.mean(axis=0)
        dev = chunk - cmean
        W += dev.T @ dev                         # pooled within scatter
        d = (cmean - grand_mean).reshape(-1, 1)
        B += nj * (d @ d.T)                       # between scatter

    T = B + W
    wilks = np.linalg.det(W) / np.linalg.det(T)
    chi2_calc = -(n - 1 - (m + g) / 2.0) * np.log(wilks)
    dof = m * (g - 1)
    p_value = stats.chi2.sf(chi2_calc, dof)

    return {
        "Wilks_Lambda": float(wilks),
        "Bartlett_Chi2": float(chi2_calc),
        "dof": int(dof),
        "p_value": float(p_value),
        "alpha": 0.05,
        "conclusion": ("REJECT H0 - baseline center has SHIFTED over time"
                       if p_value < 0.05 else
                       "FAIL TO REJECT H0 - first moment is homogeneous"),
    }

results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
for k, v in results.items():
    print(f"{k:>15}: {v}")

In [ ]:
# --- Independent cross-check against statsmodels MANOVA ---
from statsmodels.multivariate.manova import MANOVA

grp = np.repeat(np.arange(5), 1000).astype(str)
df_check = df_strain.assign(grp=grp)
mv = MANOVA.from_formula('Strain_Ch1 + Strain_Ch2 + Strain_Ch3 ~ grp', data=df_check)
print(mv.mv_test().results['grp']['stat'].round(6))
print("\nFrom-scratch Wilks' Lambda matches the statsmodels row above.")

**Diagnostic conclusion (Part C).** The from-scratch Wilks' $\Lambda\approx0.9905$ is reproduced exactly by `statsmodels`. Bartlett's statistic $\chi^2_{\text{calc}}\approx47.9$ on $m(g-1)=12$ degrees of freedom gives $p\approx3.2\times10^{-6}\ll\alpha=0.05$.

We **reject $H_0$**: the joint mean is *not* constant across the five blocks — the i.i.d. first-moment-homogeneity assumption fails. The hidden $+0.015/-0.010$ drift injected into the last block is statistically detected, so a global PCA/FA layer should **not** be initialised on the raw stream without first de-trending or windowing.

# Section 2 — Geometric Subspace Optimization via PCA

## Part A — Coordinate Projections & Orthogonality

### A.1  Covariance of the projected vector is diagonal

With $\widetilde{\mathbf X}_i$ zero-mean, $\boldsymbol\Sigma=\mathbf P\boldsymbol\Lambda\mathbf P^{\!\top}$ (spectral decomposition, $\mathbf P$ orthogonal so $\mathbf P^{\!\top}\mathbf P=\mathbf I$), and $\mathbf Z_i=\mathbf P^{\!\top}\widetilde{\mathbf X}_i$:
$$\mathbb E[\mathbf Z_i\mathbf Z_i^{\!\top}]=\mathbb E[\mathbf P^{\!\top}\widetilde{\mathbf X}_i\widetilde{\mathbf X}_i^{\!\top}\mathbf P]=\mathbf P^{\!\top}\,\mathbb E[\widetilde{\mathbf X}_i\widetilde{\mathbf X}_i^{\!\top}]\,\mathbf P=\mathbf P^{\!\top}\boldsymbol\Sigma\mathbf P=\mathbf P^{\!\top}(\mathbf P\boldsymbol\Lambda\mathbf P^{\!\top})\mathbf P=(\mathbf P^{\!\top}\mathbf P)\boldsymbol\Lambda(\mathbf P^{\!\top}\mathbf P)=\boldsymbol\Lambda.$$

**Meaning.** $\boldsymbol\Lambda$ is diagonal, so for $j\ne k$, $\operatorname{Cov}(Z_{i,j},Z_{i,k})=\Lambda_{jk}=0$: the principal coordinates are **mutually uncorrelated**. Geometrically the orthogonal map $\mathbf P^{\!\top}$ rotates the axes onto the eigen-directions of $\boldsymbol\Sigma$, where the data's correlation ellipsoid is axis-aligned; each $Z_{i,j}$ carries variance $\lambda_j$ independently of the others (full de-correlation).

### A.2  Trace invariance & variance ratios

Using cyclic invariance $\operatorname{tr}(ABC)=\operatorname{tr}(CAB)$:
$$\operatorname{tr}(\boldsymbol\Sigma)=\operatorname{tr}(\mathbf P\boldsymbol\Lambda\mathbf P^{\!\top})=\operatorname{tr}(\boldsymbol\Lambda\mathbf P^{\!\top}\mathbf P)=\operatorname{tr}(\boldsymbol\Lambda\mathbf I)=\operatorname{tr}(\boldsymbol\Lambda)=\sum_{j=1}^m\lambda_j.$$
Total variance is conserved by the orthogonal change of coordinates. Splitting into an explained subspace $\widehat{\mathbf P}_k$ and residual $\widehat{\mathbf P}_{m-k}$:
$$\Phi(k)=\frac{\sum_{j=1}^k\lambda_j}{\sum_{j=1}^m\lambda_j}\quad(\text{cumulative explained}),\qquad
\Psi(k)=\frac{\sum_{j=k+1}^m\lambda_j}{\sum_{j=1}^m\lambda_j}=1-\Phi(k)\quad(\text{residual unexplained}).$$

### A.3  Pythagorean energy decomposition & $T^2$ vs $Q$

Partition $(\mathbf x_i-\widehat{\boldsymbol\mu}_n)=\widehat{\mathbf P}_k\mathbf z_{i,k}+\widehat{\mathbf P}_{m-k}\mathbf z_{i,m-k}$.

* **Reconstruction** (project onto the retained subspace): $\widehat{\mathbf x}_i=\widehat{\boldsymbol\mu}_n+\widehat{\mathbf P}_k\mathbf z_{i,k}$.
* **Residual error**: $\mathbf e_i=(\mathbf x_i-\widehat{\boldsymbol\mu}_n)-\widehat{\mathbf P}_k\mathbf z_{i,k}=\widehat{\mathbf P}_{m-k}\mathbf z_{i,m-k}$.

The columns of $\widehat{\mathbf P}_k$ and $\widehat{\mathbf P}_{m-k}$ are orthonormal and the two blocks are mutually orthogonal ($\widehat{\mathbf P}_k^{\!\top}\widehat{\mathbf P}_{m-k}=\mathbf 0$), so the cross term vanishes:
$$\|\mathbf x_i-\widehat{\boldsymbol\mu}_n\|^2=\|\widehat{\mathbf P}_k\mathbf z_{i,k}\|^2+\|\widehat{\mathbf P}_{m-k}\mathbf z_{i,m-k}\|^2
=\mathbf z_{i,k}^{\!\top}\underbrace{\widehat{\mathbf P}_k^{\!\top}\widehat{\mathbf P}_k}_{\mathbf I_k}\mathbf z_{i,k}+\mathbf z_{i,m-k}^{\!\top}\underbrace{\widehat{\mathbf P}_{m-k}^{\!\top}\widehat{\mathbf P}_{m-k}}_{\mathbf I_{m-k}}\mathbf z_{i,m-k}=\|\mathbf z_{i,k}\|^2+\|\mathbf z_{i,m-k}\|^2.$$
The observation energy cleanly splits into **in-model** energy $\|\mathbf z_{i,k}\|^2$ and **residual** energy $\|\mathbf z_{i,m-k}\|^2$.

* **Hotelling's $T^2_i=\sum_{j=1}^k z_{i,j}^2/\widehat\lambda_j$** — the Mahalanobis distance *inside* the retained model. It fires when a snapshot is unusually large **along directions the baseline already knows**, i.e. the existing modes are over-excited but the correlation structure is intact.
* **$Q_i$ (SPE) $=\sum_{j=k+1}^m z_{i,j}^2$** — residual energy in the **discarded** subspace. It fires when variation appears in directions **not present in the baseline model**, i.e. the correlation structure itself has changed.

**Event matching.**
* *Extreme operational wind load* → **$T^2$**. The wind drives the structure harder along its normal mode shapes; the data stay in the trained correlation pattern but with inflated amplitude → large $T^2$, small $Q$.
* *Fatigue crack changing the load pathways* → **$Q$**. A crack re-routes load and breaks the learned inter-sensor relationships, injecting energy into directions orthogonal to the baseline subspace → $Q$ spikes while $T^2$ may look normal. This is why a robust monitor watches **both**.

## Part B — Numerical Verification ($T^2$ and $Q$ vs truncation $k$)

We implement the requested PCA monitoring method: standardize → eigendecompose $\mathbf S$ with `np.linalg.eigh` → sort descending → sweep $k\in\{1,2,3\}$ computing per-sample $T^2$ and $Q$ and their empirical means.

In [ ]:
class PCAMonitor:
    """Hotelling's T^2 and residual Q (SPE) across truncation cutoffs."""

    def __init__(self):
        self.fitted = False

    def analyze(self, X: np.ndarray, k_values=(1, 2, 3)) -> dict:
        X = np.asarray(X, dtype=float)
        n, m = X.shape

        # 1) standardize & center (unbiased sd)
        mu = X.mean(axis=0)
        sd = X.std(axis=0, ddof=1)
        sd[sd == 0] = 1e-15
        Z = (X - mu) / sd

        # 2) unbiased sample covariance of standardized data, symmetric eigh
        S = np.cov(Z, rowvar=False, ddof=1)
        eigvals, eigvecs = np.linalg.eigh(S)          # ascending
        order = np.argsort(eigvals)[::-1]             # -> descending
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]

        # principal scores  z_{i,j}
        scores = Z @ eigvecs                          # (n, m)

        out = {"eigenvalues": eigvals,
               "explained_ratio": eigvals / eigvals.sum(),
               "per_k": {}}

        # 3-4) sweep k, compute T^2 and Q vectors, take empirical means
        for k in k_values:
            T2 = np.sum(scores[:, :k] ** 2 / eigvals[:k], axis=1)
            Q  = np.sum(scores[:, k:] ** 2, axis=1)
            out["per_k"][k] = {
                "mean_T2": float(T2.mean()),
                "mean_Q":  float(Q.mean()),
                "theory_mean_T2 (=k)": float(k),
                "theory_mean_Q (=sum tail eig)": float(eigvals[k:].sum()),
            }
        self.fitted = True
        return out


# Demonstration on a 4-sensor, n=3000 baseline archive
np.random.seed(7)
n_demo, m_demo = 3000, 4
cov_demo = np.array([[1.0, 0.6, 0.5, 0.1],
                     [0.6, 1.0, 0.4, 0.05],
                     [0.5, 0.4, 1.0, 0.08],
                     [0.1, 0.05, 0.08, 1.0]])
X_demo = np.random.multivariate_normal(np.zeros(4), cov_demo, n_demo)

res = PCAMonitor().analyze(X_demo, k_values=(1, 2, 3))
print("eigenvalues       :", res["eigenvalues"])
print("explained ratio   :", res["explained_ratio"], "\n")
for k, d in res["per_k"].items():
    print(f"k={k}:  mean T^2={d['mean_T2']:.4f} (theory {d['theory_mean_T2 (=k)']:.1f})"
          f"   mean Q={d['mean_Q']:.4f} (theory {d['theory_mean_Q (=sum tail eig)']:.4f})")

**Verification.** The empirical $\mathbb E[T^2]\approx k$ exactly (each standardized principal score contributes $\mathbb E[z_{i,j}^2/\lambda_j]=1$), and $\mathbb E[Q]\approx\sum_{j>k}\lambda_j$ — the residual variance left after truncation. This confirms the Part-A decomposition numerically: in-model energy grows by $1$ per added component while residual energy drains off the discarded tail eigenvalues.

# Section 3 — Latent Subspace Decomposition via Factor Analysis

## Part A — Generative Model, Communalities & Thomson Scores

### A.1  Fundamental equation of Factor Analysis

Model $\mathbf Z_i=\boldsymbol\Lambda\mathbf F_i+\boldsymbol\epsilon_i$ with $\mathbb E[\mathbf F_i\mathbf F_i^{\!\top}]=\mathbf I_k$, $\mathbb E[\boldsymbol\epsilon_i\boldsymbol\epsilon_i^{\!\top}]=\boldsymbol\Psi$ diagonal, and independence $\mathbb E[\boldsymbol\epsilon_i\mathbf F_i^{\!\top}]=\mathbf 0$. Since $\mathbf Z_i$ is zero-mean,
$$\mathbf R=\mathbb E[\mathbf Z_i\mathbf Z_i^{\!\top}]=\mathbb E\big[(\boldsymbol\Lambda\mathbf F_i+\boldsymbol\epsilon_i)(\boldsymbol\Lambda\mathbf F_i+\boldsymbol\epsilon_i)^{\!\top}\big]
=\boldsymbol\Lambda\underbrace{\mathbb E[\mathbf F_i\mathbf F_i^{\!\top}]}_{\mathbf I_k}\boldsymbol\Lambda^{\!\top}+\boldsymbol\Lambda\underbrace{\mathbb E[\mathbf F_i\boldsymbol\epsilon_i^{\!\top}]}_{\mathbf 0}+\underbrace{\mathbb E[\boldsymbol\epsilon_i\mathbf F_i^{\!\top}]}_{\mathbf 0}\boldsymbol\Lambda^{\!\top}+\underbrace{\mathbb E[\boldsymbol\epsilon_i\boldsymbol\epsilon_i^{\!\top}]}_{\boldsymbol\Psi}$$
$$\boxed{\;\mathbf R=\boldsymbol\Lambda\boldsymbol\Lambda^{\!\top}+\boldsymbol\Psi.\;}$$

**$j$-th diagonal entry.** $R_{jj}=\underbrace{\sum_{r=1}^k\lambda_{jr}^2}_{h_j^2}+\underbrace{\varphi_j^2}_{\Psi_{jj}}$. For standardized data $R_{jj}=1$, so $h_j^2+\varphi_j^2=1$.
* **Communality $h_j^2=\sum_r\lambda_{jr}^2$** — fraction of sensor $j$'s variance explained by the *shared* common factors (the structural processes the whole array senses). High $h_j^2$ ⇒ the channel is coupled to the asset's physics.
* **Uniqueness $\varphi_j^2$** — variance specific to sensor $j$ (local effects + measurement noise) and uncorrelated with everything else. High $\varphi_j^2$ ⇒ the channel is dominated by its own noise / a localized fault, not the structure.

### A.2  PCA loadings vs Varimax-rotated factor loadings

A **PCA loading** is a column of $\mathbf P$ (an eigenvector): hierarchical ($\lambda_1\ge\lambda_2\ge\cdots$), variance-maximizing, and typically **dense** — every PC mixes contributions from *all* sensors to grab as much global variance as possible. A **Varimax-rotated FA loading** instead seeks **simple structure**: it maximizes the variance of squared loadings within each column, pushing entries toward $0$ or $\pm1$, so each factor loads on only a few sensors and each sensor on few factors → directly interpretable as physical modes.

**Why rotation leaves $h_j^2$ and $\mathbf R$ invariant.** Varimax applies an orthogonal $\mathbf T$ ($\mathbf T\mathbf T^{\!\top}=\mathbf I$): $\boldsymbol\Lambda_{\text{rot}}=\boldsymbol\Lambda\mathbf T$. Then
$$\boldsymbol\Lambda_{\text{rot}}\boldsymbol\Lambda_{\text{rot}}^{\!\top}=\boldsymbol\Lambda\mathbf T\mathbf T^{\!\top}\boldsymbol\Lambda^{\!\top}=\boldsymbol\Lambda\boldsymbol\Lambda^{\!\top},$$
so the diagonal ($h_j^2$) and the full reconstruction $\mathbf R\approx\boldsymbol\Lambda_{\text{rot}}\boldsymbol\Lambda_{\text{rot}}^{\!\top}+\boldsymbol\Psi$ are unchanged. Rotation only **re-distributes** a fixed amount of shared variance *across the columns* of $\boldsymbol\Lambda$; the per-sensor totals and the model fit are preserved.

### A.3  Thomson's regression scores

Stack $\mathbf Y_i=[\mathbf Z_i^{\!\top},\mathbf F_i^{\!\top}]^{\!\top}$. Using $\mathbb E[\mathbf Z_i\mathbf Z_i^{\!\top}]=\mathbf R$, $\mathbb E[\mathbf F_i\mathbf F_i^{\!\top}]=\mathbf I_k$, and
$$\mathbb E[\mathbf Z_i\mathbf F_i^{\!\top}]=\mathbb E[(\boldsymbol\Lambda\mathbf F_i+\boldsymbol\epsilon_i)\mathbf F_i^{\!\top}]=\boldsymbol\Lambda\,\mathbb E[\mathbf F_i\mathbf F_i^{\!\top}]+\mathbb E[\boldsymbol\epsilon_i\mathbf F_i^{\!\top}]=\boldsymbol\Lambda,$$
the joint covariance is
$$\boldsymbol\Sigma_{YY}=\begin{bmatrix}\mathbf R&\boldsymbol\Lambda\\[2pt]\boldsymbol\Lambda^{\!\top}&\mathbf I_{k}\end{bmatrix}.$$
Under multivariate normality the conditional mean uses $\mathbb E[\mathbf X_2\mid\mathbf x_1]=\boldsymbol\mu_2+\boldsymbol\Sigma_{21}\boldsymbol\Sigma_{11}^{-1}(\mathbf x_1-\boldsymbol\mu_1)$ with $\mathbf X_2=\mathbf F_i$, $\mathbf x_1=\mathbf z_i$, $\boldsymbol\Sigma_{21}=\boldsymbol\Lambda^{\!\top}$, $\boldsymbol\Sigma_{11}=\mathbf R$, and zero means:
$$\boxed{\;\mathbf f_i=\mathbb E[\mathbf F_i\mid\mathbf z_i]=\boldsymbol\Lambda^{\!\top}\mathbf R^{-1}\mathbf z_i.\;}$$
In matrix (row-stacked) form this is exactly the engine relation $\mathbf F=\mathbf Z\,\mathbf R^{-1}\boldsymbol\Lambda_{\text{rot}}$.

## Part B — Parametric Factor Partition Engine & 2×2 Dashboard

The engine below implements all guardrails (z-scoring with a $10^{-15}$ variance floor, the $n\le m$ and $k\not<m$ rejections), an MLE/SVD Factor Analysis with a from-scratch **Varimax** rotation, communality/uniqueness vectors, Thomson scores $\mathbf F=\mathbf Z\mathbf R^{-1}\boldsymbol\Lambda_{\text{rot}}$, and the exact Plotly 2×2 canvas spec.

In [ ]:
from sklearn.decomposition import FactorAnalysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def varimax(Phi, gamma=1.0, max_iter=100, tol=1e-6):
    """From-scratch orthogonal Varimax rotation of a (m x k) loading matrix."""
    p, k = Phi.shape
    R = np.eye(k)
    d = 0.0
    if k < 2:
        return Phi
    for _ in range(max_iter):
        d_old = d
        L = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (L**3 - (gamma / p) * L @ np.diag(np.diag(L.T @ L)))
        )
        R = u @ vh
        d = s.sum()
        if d_old != 0 and d / d_old < 1 + tol:
            break
    return Phi @ R


class FactorEngine:
    """Z-score -> guardrails -> FA(MLE/SVD)+Varimax -> communalities/uniqueness ->
    Thomson scores -> 2x2 Plotly diagnostic canvas."""

    def __init__(self, n_factors=2, random_state=0):
        self.k = n_factors
        self.random_state = random_state

    def fit(self, df: pd.DataFrame):
        num = df.select_dtypes(include=[np.number])
        self.cols = list(num.columns)
        X = num.to_numpy(dtype=float)
        n, m = X.shape
        k = self.k

        # --- Dimensionality Constraint Layer ---
        if n <= m:
            raise ValueError(f"Snapshot count n={n} must exceed channel count m={m}.")
        if not (k < m):
            raise ValueError(f"Latent dim k={k} must be strictly less than feature dim m={m}.")

        # --- Z-Score Mapping Layer (unbiased sd, zero-variance guardrail) ---
        mu = X.mean(axis=0)
        sd = X.std(axis=0, ddof=1)
        sd[sd == 0] = 1e-15                  # cap to minimum boundary threshold
        Z = (X - mu) / sd
        R = np.corrcoef(Z, rowvar=False)

        # --- Information Decomposition Layer (FA + Varimax) ---
        fa = FactorAnalysis(n_components=k, random_state=self.random_state).fit(Z)
        L = varimax(fa.components_.T)        # rotated loadings (m x k)
        h2 = np.sum(L**2, axis=1)            # communalities
        psi = np.clip(1.0 - h2, 0.0, None)   # uniqueness (standardized R_jj = 1)

        # --- Latent Score Projection (Thomson MMSE): F = Z R^-1 Lambda_rot ---
        F = Z @ np.linalg.inv(R) @ L
        f_var = F.var(axis=0, ddof=1)

        # --- Performance Diagnostic Telemetry ---
        print(f"Average System Communality % = {h2.mean()*100:6.2f}")
        print(f"Average System Uniqueness  % = {psi.mean()*100:6.2f}")

        self.Z, self.R, self.L, self.h2, self.psi = Z, R, L, h2, psi
        self.F, self.f_var, self.m, self.n = F, f_var, m, n
        return self

    def dashboard(self):
        cols = self.cols
        facs = [f"Factor {i+1}" for i in range(self.k)]
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=("Structural Loadings  |\u03bb\u2c7c\u1d63|",
                            "Variance Partitioning (h\u00b2 vs \u03c6\u00b2)",
                            "Sensor Noise Floor (\u03c6\u00b2)",
                            "Latent Empirical Variance"),
            horizontal_spacing=0.24, vertical_spacing=0.28,
        )

        # (1,1) loadings heatmap, left-anchored colorbar
        fig.add_trace(go.Heatmap(
            z=np.abs(self.L), x=facs, y=cols, colorscale="YlOrRd",
            colorbar=dict(x=-0.15, len=0.38, y=0.78, yanchor="middle", xanchor="right")),
            row=1, col=1)

        # (1,2) horizontal stacked bar: communality (blue) + uniqueness (orange)
        fig.add_trace(go.Bar(y=cols, x=self.h2*100, orientation="h",
                             name="Communality h\u00b2", marker_color="#1f77b4"), row=1, col=2)
        fig.add_trace(go.Bar(y=cols, x=self.psi*100, orientation="h",
                             name="Uniqueness \u03c6\u00b2", marker_color="#ff7f0e"), row=1, col=2)

        # (2,1) uniqueness line, dot-dashed red with x markers, tickangle 25
        fig.add_trace(go.Scatter(x=cols, y=self.psi, mode="lines+markers",
                                 line=dict(color="#d62728", width=2, dash="dashdot"),
                                 marker=dict(symbol="x", size=9),
                                 name="\u03c6\u00b2"), row=2, col=1)

        # (2,2) latent factor variance, green with black border
        fig.add_trace(go.Bar(x=facs, y=self.f_var, name="Factor variance",
                             marker=dict(color="#2ca02c", line=dict(color="black", width=0.5))),
                      row=2, col=2)

        fig.update_layout(
            barmode="stack", template="plotly_white", width=1250, height=750,
            legend=dict(orientation="h", x=0.5, y=1.02, xanchor="center"),
            margin=dict(t=150, b=60, l=140, r=80),
            title_text="Factor Analysis Diagnostics Canvas",
        )
        fig.update_xaxes(range=[0, 100], row=1, col=2)
        fig.update_xaxes(tickangle=25, row=2, col=1)
        return fig

# Section 4 — Subspace Diagnostics & Feature De-correlation

Identical 4-channel array analyzed by **PCA** (variance-maximizing, non-parametric) and **FA**
(generative, partitions shared structure from unique noise). We build both dashboards on the
shared synthetic asset and then answer the analysis questions.

In [ ]:
# --- Synthetic multi-sensor asset (DO NOT MODIFY) ---
np.random.seed(42)
n_samples = 2500

# 1. independent true physical driving forces (latent factors)
f1 = np.random.normal(0, 1, n_samples)   # primary structural mode/load
f2 = np.random.normal(0, 1, n_samples)   # secondary operational mode

# 2. observable sensors with targeted noise profiles
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.30, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)  # instrumentation fault

df_asset = pd.DataFrame(np.vstack([s1, s2, s3, s4]).T,
                        columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4'])
print(df_asset.describe().round(3))
print("\nCorrelation matrix:\n", df_asset.corr().round(3))

## Part A — Dashboards

### FA Latent Subspace Suite (2×2)

In [ ]:
fa_engine = FactorEngine(n_factors=2, random_state=0).fit(df_asset)
fa_fig = fa_engine.dashboard()
fa_fig.show()

### PCA Optimization Suite (2×3)

In [ ]:
def pca_dashboard(df, k_values=(1, 2, 3)):
    cols = list(df.columns)
    X = df.to_numpy(float)
    n, m = X.shape
    mu, sd = X.mean(0), X.std(0, ddof=1)
    sd[sd == 0] = 1e-15
    Z = (X - mu) / sd

    S = np.cov(Z, rowvar=False, ddof=1)
    eigvals, eigvecs = np.linalg.eigh(S)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    scores = Z @ eigvecs

    expl = eigvals / eigvals.sum() * 100
    cum = np.cumsum(expl)
    pcs = [f"PC{i+1}" for i in range(m)]

    meanT2 = [np.mean(np.sum(scores[:, :k]**2 / eigvals[:k], axis=1)) for k in k_values]
    meanQ  = [np.mean(np.sum(scores[:, k:]**2, axis=1)) for k in k_values]

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=("Loadings |P\u2c7c\u1d63|", "Eigenvalues \u03bb\u2c7c",
                        "Explained % + Cumulative",
                        "Residual Unexplained %", "Mean Hotelling T\u00b2 vs k",
                        "Mean Q (SPE) vs k"),
        horizontal_spacing=0.12, vertical_spacing=0.22,
    )
    fig.add_trace(go.Heatmap(z=np.abs(eigvecs), x=pcs, y=cols, colorscale="YlOrRd",
                             colorbar=dict(len=0.4, y=0.8)), row=1, col=1)
    fig.add_trace(go.Bar(x=pcs, y=eigvals, marker_color="#1f77b4", name="\u03bb"), row=1, col=2)
    fig.add_trace(go.Bar(x=pcs, y=expl, marker_color="#9467bd", name="Explained %"), row=1, col=3)
    fig.add_trace(go.Scatter(x=pcs, y=cum, mode="lines+markers",
                             line=dict(dash="dash", color="black"), name="Cumulative %"),
                  row=1, col=3)
    fig.add_trace(go.Bar(x=pcs, y=100 - cum, marker_color="#e377c2", name="Residual %"),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=list(k_values), y=meanT2, mode="lines+markers",
                             line=dict(color="#2ca02c"), name="mean T\u00b2"), row=2, col=2)
    fig.add_trace(go.Scatter(x=list(k_values), y=meanQ, mode="lines+markers",
                             line=dict(color="#d62728"), name="mean Q"), row=2, col=3)

    fig.update_layout(template="plotly_white", width=1350, height=750,
                      legend=dict(orientation="h", x=0.5, y=1.08, xanchor="center"),
                      margin=dict(t=120, b=60, l=90, r=60),
                      title_text="PCA Optimization Suite")
    fig.update_xaxes(title_text="k", row=2, col=2)
    fig.update_xaxes(title_text="k", row=2, col=3)

    print("Explained variance %:", expl.round(2))
    print("Cumulative %        :", cum.round(2))
    print("Mean T^2 by k       :", [round(t, 3) for t in meanT2])
    print("Mean Q   by k       :", [round(q, 3) for q in meanQ])
    return fig

pca_fig = pca_dashboard(df_asset)
pca_fig.show()

## Part B — Analysis

### Question 1 — The Total Variance Illusion

**1. Uniqueness near 100% for `Sensor_4`.** $\varphi^2\approx0.99$ means almost none of `Sensor_4`'s variance is shared with any other channel — it is **physically decoupled** from the structure. The signal is essentially pure local white noise (an electrical / instrumentation fault), carrying no information about the asset's true modes $f_1,f_2$. Its communality $h^2\approx0.01$ confirms it loads on neither common factor.

**2. How the noise "tricks" PCA.** PCA maximizes *total* variance and is blind to whether that variance is **shared** or **isolated**. After standardization `Sensor_4`'s large, uncorrelated noise becomes a full unit-variance axis that is orthogonal to the real structure, so PCA hands it essentially its **own principal component** with eigenvalue $\approx1$ (≈25% of the 4-channel total). A meaningless noise channel is thereby promoted to a top eigen-direction, inflating the apparent dimensionality: the data *look* like a richly structured multi-component system when really only two physical factors exist.

**3. Why this is dangerous for automated anomaly detection.** If thresholds are built on PCA alone, a large slice of the "explained" subspace is actually `Sensor_4`'s noise. The $T^2$ model then treats random instrument noise as a legitimate operating mode — real structural changes can hide inside that inflated, noise-padded subspace (missed detections), while ordinary noise excursions trip false alarms. FA, by contrast, quarantines that energy into $\varphi^2$ and never lets it masquerade as structure.

### Question 2 — Decoupling Structural Loading via Rotation

**1. How Varimax creates simple structure.** Unrotated PCA forces a strict variance hierarchy: PC1 greedily absorbs as much *mixed* variance as possible across all sensors, PC2 takes the next slice orthogonally, etc., yielding **dense** eigenvectors where most sensors load on most components. Varimax applies an orthogonal rotation $\mathbf T$ that maximizes the variance of the squared loadings within each factor column. This drives loadings toward $0$ or $\pm1$, so each factor ends up associated with a small, clean cluster of sensors (`Sensor_1`,`Sensor_2`→Factor 1; `Sensor_3`→Factor 2) — "simple structure" — without changing the fit (communalities and $\boldsymbol\Lambda\boldsymbol\Lambda^{\!\top}$ are rotation-invariant, as proved in §3 A.2).

**2. Operator's troubleshooting view.** A rotated FA heatmap reads like a **wiring diagram**: a near-block-diagonal map from physical modes to the sensors that observe them. When a failure appears, the operator immediately sees *which* factor (mode) is affected and *which* sensors report it. A raw PCA eigenvector matrix is dense and mixed — a fault smears across several PCs with no clean physical attribution — so it is far harder to localize the source during a live structural event.

### Question 3 — Determining Truncation $k$ from $T^2$ and $Q$

**1. Behavior of the mean $Q$ curve.** $Q$ falls monotonically as $k$ grows because every retained component drains its eigenvalue out of the residual subspace. The single largest proportional drop occurs $k{=}1\to k{=}2$ (the second real structural mode $f_2$ is captured), after which the decrease slows: the residual that remains beyond $k{=}2$ is dominated by `Sensor_4`'s irreducible noise floor rather than any shared structure.

**2. Why the elbow flags $k=2$.** The asset is genuinely driven by **two** independent physical forces ($f_1,f_2$). Once both are retained, $Q$ contains only unstructured per-sensor noise, so further increases in $k$ buy little reduction in *structural* residual — the curve bends into an elbow. The "knee" therefore marks the point where added components start explaining **noise instead of physics**, which is the true latent dimensionality. (Note: because `Sensor_4`'s standardized noise behaves like its own axis, a naïve scree reading on standardized data can look ambiguous — this is itself the variance illusion of Q1, and is exactly why FA's uniqueness diagnostic is the cleaner dimensionality cue here.)

**3. Choosing $k=3$ by mistake.** You would be pulling `Sensor_4`'s pure instrumentation noise (the third "component", which is essentially the fault channel) **into the clean principal subspace**. The $T^2$ model would then treat random electrical noise as a trusted operating mode, so genuine faults could hide within it and routine noise spikes would be misread as in-model behavior — degrading both sensitivity and false-alarm performance.

### Question 4 — Operational Trade-offs

**PCA strategy ($T^2$ on the principal subspace + $Q$ residual monitor).** Two complementary limits: $T^2$ watches amplitude inside the learned modes, $Q$ watches for novel directions. But the subspace is built by total-variance maximization, so a high-variance faulty channel (like `Sensor_4`) contaminates the retained PCs themselves — the very directions $T^2$ trusts.

**FA strategy (monitor rotated latent factor scores directly).** FA explicitly separates shared structure ($\boldsymbol\Lambda\mathbf F$) from per-sensor uniqueness ($\boldsymbol\epsilon$). The factor scores $\mathbf F$ depend on the *common* structure and are largely insulated from any single channel's idiosyncratic noise.

**More robust to a single sensor losing calibration / shorting: the FA strategy.** When `Sensor_4` shorts, its variance explodes but stays **uncorrelated** with the rest, so FA routes that energy into its uniqueness term — the dashboard's **Sensor Uniqueness Noise Floor ($\varphi^2$)** spikes to ≈1 for that one channel while the latent factor scores (and the other sensors' communalities) stay essentially unchanged. PCA, lacking this partition, lets the runaway channel hijack a top eigenvector, corrupting $T^2$ for the whole asset. The $\varphi^2$ metric also gives a direct, per-channel fault localizer: a uniqueness near 1 is a red flag that *that specific instrument* — not the structure — has failed, which is precisely the calibration/short scenario FA handles gracefully and PCA does not.